In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

def prepare_data(base_path):
    print("Loading data...")
    train = pd.read_csv(base_path + 'train.csv')
    test = pd.read_csv(base_path + 'test.csv')
    stores = pd.read_csv(base_path + 'stores.csv')
    oil = pd.read_csv(base_path + 'oil.csv')
    holidays = pd.read_csv(base_path + 'holidays_events.csv')
    transactions = pd.read_csv(base_path + 'transactions.csv')

    # Date conversion
    for df in [train, test, oil, holidays, transactions]:
        df['date'] = pd.to_datetime(df['date'])

    # Filter train data to speed up and focus on recent trends
    train = train[train['date'] >= '2017-01-01']

    # Log transformation for target
    train['sales'] = np.log1p(train['sales'])

    # Oil processing
    oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(limit_direction='both')
    oil['oil_l1'] = oil['dcoilwtico'].shift(1)
    oil['oil_roll7'] = oil['dcoilwtico'].rolling(7).mean()

    # Holidays processing
    holidays = holidays[holidays['transferred'] == False]
    holidays = holidays.drop_duplicates(subset=['date'])

    # Combine train and test for feature engineering
    df = pd.concat([train, test], axis=0)
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    df = df.merge(holidays[['date', 'type', 'locale', 'locale_name']], on='date', how='left')

    # Time features
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['dayofmonth'] = df['date'].dt.day
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_payday'] = ((df['dayofmonth'] == 15) | (df['date'].dt.is_month_end)).astype(int)

    # Label Encoding
    le = LabelEncoder()
    cat_cols = ['family', 'city', 'state', 'type_x', 'type_y', 'locale', 'locale_name']
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))

    # Feature engineering: Lags and Rolling
    # To avoid the 2.56 score bug, we use lags that are at least 16 days old for the direct model,
    # or use recursive forecasting. For simplicity and stability, we use lags >= 16 days.
    print("Creating lag features...")
    for lag in [16, 21, 30, 60, 365]:
        df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].transform(lambda x: x.shift(lag))

    for window in [7, 30]:
        df[f'roll_mean_{window}'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
            lambda x: x.shift(16).rolling(window).mean()
        )

    # Split back to train and test
    train_df = df[df['date'] < '2017-08-16']
    test_df = df[df['date'] >= '2017-08-16']

    features = [col for col in df.columns if col not in ['id', 'date', 'sales', 'transactions']]
    
    return train_df, test_df, features

def train_model(train_df, features):
    print("Training model...")
    # Time-based split for validation
    val_start = '2017-08-01'
    X_train = train_df[train_df['date'] < val_start][features]
    y_train = train_df[train_df['date'] < val_start]['sales']
    X_val = train_df[train_df['date'] >= val_start][features]
    y_val = train_df[train_df['date'] >= val_start]['sales']

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'n_estimators': 2000,
        'early_stopping_round': 100,
        'random_state': 42
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    
    return model

def main():
    # Note: Paths are for Kaggle environment
    base_path = '/kaggle/input/store-sales-time-series-forecasting/'
    
    # train_df, test_df, features = prepare_data(base_path)
    # model = train_model(train_df, features)
    # preds = model.predict(test_df[features])
    # test_df['sales'] = np.expm1(preds)
    # test_df[['id', 'sales']].to_csv('submission.csv', index=False)
    
    print("Full pipeline defined. Use this script in Kaggle.")

if __name__ == "__main__":
    main()


Full pipeline defined. Use this script in Kaggle.
